In [4]:
import hashlib
import json
from datetime import datetime

class Transaction:
    def __init__(self, sender, receiver, amount):
        self.sender = sender
        self.receiver = receiver
        self.amount = amount
        self.timestamp = datetime.now().strftime("%H:%M:%S")
        self.tx_id = self.calculate_hash()

    def calculate_hash(self):
        # Unique fingerprint for every transaction
        hash_string = f"{self.sender}{self.receiver}{self.amount}{self.timestamp}"
        return hashlib.sha256(hash_string.encode()).hexdigest()

    def __repr__(self):
        return f"[{self.timestamp}] {self.sender} -> {self.receiver}: ${self.amount}"

In [5]:
class SimpleBlockchain:
    def __init__(self):
        self.chain = []
        self.pending_transactions = []
        # Initial balances (UTXO Pool)
        self.utxo_pool = {"Alice": 100, "Bob": 50, "Charlie": 10}

    def process_transaction(self, sender, receiver, amount):
        print(f"\n--- Processing: {sender} wants to send ${amount} ---")
        
        # DOUBLE-SPENDING PREVENTION: Check current unspent balance
        available_balance = self.utxo_pool.get(sender, 0)
        
        if available_balance >= amount:
            # 1. Deduct immediately (Locks the funds)
            self.utxo_pool[sender] -= amount
            
            # 2. Record the transaction
            tx = Transaction(sender, receiver, amount)
            self.pending_transactions.append(tx)
            
            # 3. Credit the receiver
            self.utxo_pool[receiver] = self.utxo_pool.get(receiver, 0) + amount
            
            print(f"✅ SUCCESS: Transaction {tx.tx_id[:10]}... added to mempool.")
            print(f"Current internal ledger: {self.utxo_pool}")
            return True
        else:
            print(f"❌ REJECTED: Double-spending attempt detected!")
            print(f"Error: {sender} tried to spend ${amount} but only has ${available_balance}.")
            return False

    def mine_block(self):
        if not self.pending_transactions:
            print("\nNo transactions to mine.")
            return
        
        # Group pending tx into a block
        block = {
            "index": len(self.chain) + 1,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "transactions": [str(tx) for tx in self.pending_transactions],
            "prev_hash": self.chain[-1]['hash'] if self.chain else "0"
        }
        block['hash'] = hashlib.sha256(json.dumps(block).encode()).hexdigest()
        self.chain.append(block)
        self.pending_transactions = []
        print(f"\n📦 Block #{block['index']} Mined & added to Blockchain!")

In [6]:
# Initialize our network
network = SimpleBlockchain()
print("Starting simulation...")
print(f"Initial Balances: {network.utxo_pool}")

# 1. Alice sends a valid amount
network.process_transaction("Alice", "Bob", 70)

# 2. Alice tries to DOUBLE-SPEND 
# (She started with 100, spent 70, so she only has 30 left)
# This 40 dollar request should be caught and blocked.
network.process_transaction("Alice", "Charlie", 40)

# 3. Mine the valid transaction into a block
network.mine_block()

print("\n--- FINAL BLOCKCHAIN STATE ---")
for block in network.chain:
    print(f"Index {block['index']} Hash: {block['hash'][:20]}...")

Starting simulation...
Initial Balances: {'Alice': 100, 'Bob': 50, 'Charlie': 10}

--- Processing: Alice wants to send $70 ---
✅ SUCCESS: Transaction da1a6b34c6... added to mempool.
Current internal ledger: {'Alice': 30, 'Bob': 120, 'Charlie': 10}

--- Processing: Alice wants to send $40 ---
❌ REJECTED: Double-spending attempt detected!
Error: Alice tried to spend $40 but only has $30.

📦 Block #1 Mined & added to Blockchain!

--- FINAL BLOCKCHAIN STATE ---
Index 1 Hash: 3644bc8f82699091efa1...
